In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact_manual
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
from RMT import stable_dist_sample, q_star_MC
from analytical_dos import get_chi_samples, dos

In [ ]:
PHI = {
    'tanh':   (torch.tanh,   lambda h: 1.0 - torch.tanh(h) ** 2),
    'relu':   (torch.relu,   lambda h: (h > 0).float()),
    'linear': (lambda h: h,  lambda h: torch.ones_like(h)),
}

def empirical_svdvals(alpha, sigma_W, phi_prime, N, q_star, seed=0):
    """Singular values of one realisation of the fixed-point pre-activation Jacobian."""
    torch.manual_seed(seed)
    W    = sigma_W * (2 * N) ** (-1.0 / alpha) * stable_dist_sample(alpha, size=(N, N))
    xi   = stable_dist_sample(alpha, scale=2.0 ** (-1.0 / alpha), size=N)
    h_st = float(q_star) ** (1.0 / alpha) * xi
    chi  = phi_prime(h_st)          # (N,)
    A    = W * chi[None, :]         # A_ij = W_ij chi_j
    return torch.linalg.svdvals(A).cpu().numpy()

In [ ]:
@interact_manual(
    alpha    = widgets.FloatSlider(value=1.5,   min=1.05, max=1.95,   step=0.05,  continuous_update=False),
    sigma_W  = widgets.FloatSlider(value=1.0,   min=0.2,  max=3.0,    step=0.1,   continuous_update=False),
    N        = widgets.IntSlider  (value=200,   min=50,   max=500,    step=50,    continuous_update=False),
    phi_name = widgets.Dropdown   (options=list(PHI.keys()), value='tanh'),
    n_s      = widgets.IntSlider  (value=20,    min=10,   max=80,     step=5,     continuous_update=False),
    n_fp     = widgets.IntSlider  (value=5000,  min=1000, max=20000,  step=1000,  continuous_update=False),
    n_chi    = widgets.IntSlider  (value=10000, min=1000, max=100000, step=10000, continuous_update=False),
    n_r      = widgets.IntSlider  (value=50,    min=10,   max=200,    step=10,    continuous_update=False),
)
def compute_and_plot(alpha, sigma_W, N, phi_name, n_s, n_fp, n_chi, n_r):
    phi, phi_prime = PHI[phi_name]

    print('Computing chi samples ...', end=' ', flush=True)
    chi    = get_chi_samples(alpha, sigma_W, phi=phi, phi_prime=phi_prime, n=n_chi)
    q_star = q_star_MC(alpha, sigma_W, phi=phi)[-1]
    print('done.')

    print(f'Computing empirical SVD ({n_r} realisations) ...', end=' ', flush=True)
    all_svdvals = np.concatenate([
        empirical_svdvals(alpha, sigma_W, phi_prime, N, q_star, seed=k)
        for k in range(n_r)
    ])
    svdvals = empirical_svdvals(alpha, sigma_W, phi_prime, N, q_star, seed=0)
    s_max   = np.percentile(all_svdvals, 99)
    s_grid  = np.linspace(1e-3, s_max, n_s)
    print('done.')

    print('Solving fixed-point equations (vectorised) ...', flush=True)
    rho_sv = dos(s_grid, alpha, sigma_W, chi,
                 n_samples=n_fp, n_iter=200, tol=1e-3, verbose=True)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    bin_edges = np.linspace(0, svdvals.max(), 60)
    bin_w     = np.diff(bin_edges)

    for ax, (yscale, title) in zip(axes[:2], [('linear', 'bulk'), ('log', 'log scale')]):
        counts, _ = np.histogram(svdvals, bins=bin_edges, density=True)
        ax.bar(bin_edges[:-1], counts, width=bin_w, align='edge',
               color='steelblue', alpha=0.5, label=f'empirical  N={N}')
        ax.plot(s_grid, rho_sv, 'r-o', ms=4, lw=1.5, label='analytical (C&B)')
        ax.set_xlabel('singular value $s$')
        ax.set_ylabel('density')
        ax.set_title(f'$\\alpha$={alpha}  $\\sigma_W$={sigma_W}  $\\phi$={phi_name} — {title}')
        ax.set_yscale(yscale)
        ax.set_xlim(0, s_max * 1.05)
        ax.legend()

    # CCDF tail panel
    ax       = axes[2]
    s_sorted = np.sort(all_svdvals)[::-1]
    n_total  = len(s_sorted)
    ccdf     = np.arange(1, n_total + 1) / n_total   # P(S > s_sorted[i-1])

    # restrict to top 20% to focus on tail (avoid bulk)
    tail_mask = ccdf < 0.20
    ax.loglog(s_sorted[tail_mask], ccdf[tail_mask],
              color='steelblue', lw=1.0, alpha=0.8, label=f'empirical CCDF  ({n_r} real.)')

    # reference slope −alpha anchored at median of tail region
    s_ref = np.percentile(s_sorted[tail_mask], 50)
    c_ref = ccdf[tail_mask][np.searchsorted(-s_sorted[tail_mask], -s_ref)]
    s_line = np.array([s_sorted[tail_mask].min(), s_sorted[tail_mask].max()])
    ax.loglog(s_line, c_ref * (s_line / s_ref) ** (-alpha),
              'r--', lw=1.5, label=f'slope $-\\alpha = -{alpha}$')

    ax.set_xlabel('singular value $s$')
    ax.set_ylabel('$P(S > s)$')
    ax.set_title(f'CCDF tail  ($\\alpha$={alpha}  $\\phi$={phi_name})')
    ax.legend()

    plt.tight_layout()
    plt.show()